# Training on field audio: does closing the domain gap help?

The baseline trained on clean focal clips and scored 0.81 on field soundscapes. This notebook tests how much of that ceiling was the clean-to-noisy domain shift, by training the same classifier on the field embeddings themselves instead of focal clips.

To keep this honest, the model is tested only on sites it never saw in training. Whole sites are held out, so a high score cannot come from the model recognising a familiar recording location. This is the site-based split decided earlier: it measures whether the model works somewhere genuinely new.

In [1]:
import numpy as np
import json
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# the field embeddings and their labels/sites (evaluation data from the baseline)
ss_emb = np.load('../data/soundscape_embeddings.npy')
with open('../data/full_meta.json') as f:
    meta = json.load(f)
ss_labels = meta['labels']
ss_sites = np.array(meta['sites'])

# same 47-species label space as the baseline
focal_labels = json.load(open('../data/focal_labels.json'))
species = sorted(set(focal_labels))
mlb = MultiLabelBinarizer(classes=species)
mlb.fit([species])
Y = mlb.transform(ss_labels)

print("field embeddings:", ss_emb.shape)
print("sites:", sorted(set(ss_sites)))

field embeddings: (1478, 1536)
sites: [np.str_('S03'), np.str_('S08'), np.str_('S09'), np.str_('S13'), np.str_('S15'), np.str_('S18'), np.str_('S19'), np.str_('S22'), np.str_('S23')]


c:\dev\biodiversity\mapping-biodiversity-from-sound\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:1016: UserWarning: unknown class(es) ['1491113', '25073', '47158son01', '47158son02', '47158son03', '47158son04', '47158son05', '47158son06', '47158son07', '47158son08', '47158son09', '47158son10', '47158son11', '47158son12', '47158son13', '47158son14', '47158son15', '47158son16', '47158son17', '47158son18', '47158son19', '47158son20', '47158son21', '47158son22', '47158son23', '47158son24', '47158son25', '517063'] will be ignored
  warnings.warn(


## Holding out sites for testing

The two most data-rich sites after S22 are held out as the test set, and S22 is kept in training since it holds most of the data. The classifier never sees the held-out sites during training, so its score on them reflects genuine generalisation to new locations rather than memorised context.

In [2]:
TEST_SITES = ['S15', 'S23']   # held out entirely; reasonably sampled, not S22-dominated

test_mask = np.isin(ss_sites, TEST_SITES)
train_mask = ~test_mask

X_tr, Y_tr = ss_emb[train_mask], Y[train_mask]
X_te, Y_te = ss_emb[test_mask], Y[test_mask]

print("train segments:", X_tr.shape[0], "from", sorted(set(ss_sites[train_mask])))
print("test segments:", X_te.shape[0], "from", TEST_SITES)

train segments: 1310 from [np.str_('S03'), np.str_('S08'), np.str_('S09'), np.str_('S13'), np.str_('S18'), np.str_('S19'), np.str_('S22')]
test segments: 168 from ['S15', 'S23']


## Training on field audio and scoring on held-out sites

The same one-vs-rest logistic regression as the baseline is trained, but on field embeddings from the training sites rather than focal clips. It is then scored on the two held-out sites, using only species that have both present and absent examples there, so the comparison to the baseline is like for like.

In [3]:
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

clf = OneVsRestClassifier(
    LogisticRegression(max_iter=1000, class_weight='balanced'),
    n_jobs=-1,
)
clf.fit(X_tr_s, Y_tr)
prob = clf.predict_proba(X_te_s)

# score only species scoreable in the held-out set
pos = Y_te.sum(axis=0)
mask = (pos > 0) & (pos < Y_te.shape[0])

aucs = [roc_auc_score(Y_te[:, i], prob[:, i]) for i in np.where(mask)[0]]
aps = [average_precision_score(Y_te[:, i], prob[:, i]) for i in np.where(mask)[0]]

print("species scored on held-out sites:", int(mask.sum()))
print("field-trained macro ROC-AUC:", round(np.mean(aucs), 4))
print("field-trained macro avg precision:", round(np.mean(aps), 4))

species scored on held-out sites: 17
field-trained macro ROC-AUC: 0.5567
field-trained macro avg precision: 0.1861


## Result: training on field audio does not beat the focal baseline

Trained on 1,310 field segments from seven sites and tested on two held-out sites (S15, S23), the field-trained classifier scored a macro ROC-AUC of 0.56 across 17 scoreable species, close to chance and well below the focal-trained baseline's 0.81.

The reason is data, not method. The labelled field audio is small (1,478 segments), heavily dominated by one site, and thinly spread across species, so most species have too few clear field examples to learn from. The clean focal clips, despite belonging to a different acoustic domain, provide a far stronger per-species training signal.

This is itself a finding. It shows the focal-trained baseline is not merely a convenient default but the better choice given the available data, and it demonstrates that the value of Perch lies in its large-scale pretraining having already bridged the clean-to-field domain gap, which the limited field data here is insufficient to bridge directly.